In [ ]:
# Some basics
import os
import pandas as pd
import numpy as np
from tqdm import tqdm

# Declare filenames
ROOT_DATA_DIR = (r"~\Data\Eyetracking_03_Matched_frames") # folder where the Input data sits
DATA_DIR_OUTPUT =  (r"~\Data\Eyetracking_04_Match_Segmentation")
SEMANTIC_SEGMENTATION_DIR = (r'~\Data\Semantic_01_Segmentation\Robust_batch')

# Define the column names for each type of eyetracking data
fixation_columns = ['tStart',	'tEnd',	'duration',	'xAvg',	'yAvg',	'pupilAvg',	'Movie', 'matched_frame_IDx_start',	'matched_frame_IDx_end']


# Define a dictionary to map movie labels to their corresponding video files
movie_files = {
    "movie_01": "Charite",
    "movie_02": "Ziemlich_Beste_Freunde",
    "movie_03": "High_Seas",
    "movie_04": "Biohackers",
    "movie_05": "Downton_Abbey",
    "movie_06": "New_Amsterdam"
}

# Define a mapping of eyetracking types to coordinates
coordinates_mapping = { "Fixation": ['xAvg', 'yAvg'],}

frame_width = 1921
frame_height = 1081


# Iterate through each eye tracking file
for participant_folder in os.listdir(ROOT_DATA_DIR):
    
    # Construct the full path to the participant subfolder
    participant_folder_path = os.path.join(ROOT_DATA_DIR, participant_folder)
    #print('Participant Directory:', participant_folder_path)

    # Check if the subfolder is a directory
    if os.path.isdir(participant_folder_path):

        # Filter the eyetracking files based on the presence of "fixation" in their filenames
        fixation_files = [eyetracking_file for eyetracking_file in os.listdir(participant_folder_path) if 'Fixation' in eyetracking_file]
        
        
        for eyetracking_file in fixation_files:  # Assuming your eye tracking files have a .csv extension
                      
            # Construct the output file path
            output_file = os.path.join(DATA_DIR_OUTPUT, f'{eyetracking_file[:-19]}_Pixel_Category.csv')
            output_file_path = os.path.join(DATA_DIR_OUTPUT, output_file)
            #print(f'OutputFile', output_file)

            # Check if the output file already exists
            if os.path.exists(output_file_path):
                #print(f"Output file already exists for {eyetracking_file}. Skipping to the next file.")
                continue
            
            file_path = os.path.join(participant_folder_path, eyetracking_file)
            movie_name = None

            # Extract the information from the eyetracking file name
            file_parts = eyetracking_file[:-3].split("_")
            participant = file_parts[0] 
            block_number = file_parts[-7]
            order_number = file_parts[-5]
            eyetracking_type = file_parts[-9]

            # Load the eye tracking data
            eyetracking_data = pd.read_csv((file_path), delimiter=',')
            print(f'File Loaded: {file_path} ...')

            # Load the eye tracking data
            eyetracking_data = pd.read_csv((file_path), delimiter=',')
            eyetracking_data = eyetracking_data[fixation_columns]
            eyetracking_data[['xAvg', 'yAvg']] = eyetracking_data[['xAvg', 'yAvg']].apply(np.floor)

            
            # Extract the desired columns from the eyetracking data if they exist in the file
            eyetracking_data = eyetracking_data[fixation_columns]
            movie_name = eyetracking_data['Movie'].loc[0]
            #print(movie_name)

            eyetracking_data['OutOfBounds'] = ((eyetracking_data['xAvg'] < 0) | (eyetracking_data['xAvg'] >= frame_width) | 
                                                (eyetracking_data['yAvg'] < 0) | (eyetracking_data['yAvg'] >= frame_height))

            
            for mask_folder in os.listdir(SEMANTIC_SEGMENTATION_DIR):
                if mask_folder.startswith(movie_files[movie_name][0]):
                    mask_folder_path = os.path.join(SEMANTIC_SEGMENTATION_DIR, mask_folder)
                    total_frames = len(os.listdir(mask_folder_path))
                    
                    for index, row in tqdm(eyetracking_data.iterrows(), total=eyetracking_data.shape[0]):

                        if row['OutOfBounds']:
                            continue  # Skip this row if the coordinate is out-of-bounds

                        frame_index_start = row['matched_frame_IDx_start']
                        frame_index_end = row['matched_frame_IDx_end']

                        # Skip this row if the frame index is out-of-bounds
                        if (frame_index_start < 0 or frame_index_start >= (total_frames/2)) or (frame_index_end < 0 or frame_index_end >= (total_frames/2)):
                            continue

                        #print("Fixations found...")
                        frame_index_start = row['matched_frame_IDx_start']
                        frame_index_end = row['matched_frame_IDx_end']
                        xAvg = row['xAvg']
                        yAvg = row['yAvg']

                        # Convert averages to integer indices
                        x_index_start = int(xAvg)
                        y_index_start = int(yAvg)
                        x_index_end = int(xAvg)
                        y_index_end = int(yAvg)

                        # Load the corresponding frame_data for start frame index
                        frame_filename_start = f"{mask_folder}_frame_{frame_index_start:04}.csv"
                        frame_file_path_start = os.path.join(mask_folder_path, frame_filename_start)
                        frame_data_start = pd.read_csv(frame_file_path_start, delimiter=';', header=None)
                        
                        # Load the corresponding frame_data for end frame index
                        frame_filename_end = f"{mask_folder}_frame_{frame_index_end:04}.csv"
                        frame_file_path_end = os.path.join(mask_folder_path, frame_filename_end)
                        frame_data_end = pd.read_csv(frame_file_path_end, delimiter=';', header=None)

                        # Get the matching pixels from frame_data_start and frame_data_end
                        matching_pixel_start = frame_data_start.iloc[y_index_start, x_index_start].astype(int)
                        matching_pixel_end = frame_data_end.iloc[y_index_end, x_index_end].astype(int)
                        
                        # Update the corresponding columns in the eyetracking data
                        eyetracking_data.at[index, 'MatchingPixel_Start'] = matching_pixel_start
                        eyetracking_data.at[index, 'MatchingPixel_End'] = matching_pixel_end

            eyetracking_data.to_csv(output_file_path)
            print('File Saved')
            print("Participant ", participant, "Block", block_number, "Order", order_number)
            print("------------------------------")
